# Capital Gain Case (Track A): Real Public Data

This notebook uses prepared data from UK Land Registry Price Paid Data.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for s in ['seaborn-v0_8-whitegrid','seaborn-whitegrid','ggplot']:
    try:
        plt.style.use(s)
        break
    except Exception:
        pass

cands=[
    Path('examples/capital_gain/data/capital_gain_dataset.csv'),
    Path('data/capital_gain_dataset.csv'),
]
csv=next((x for x in cands if x.exists()), None)
if csv is None:
    raise FileNotFoundError('Run prepare_capital_gain_data.py first.')

df=pd.read_csv(csv)
df['month']=pd.to_datetime(df['month'], errors='coerce')
print('Dataset:', csv.resolve())
print('Rows:', len(df), 'Panels:', df['panel_id'].nunique())
print('Date range:', df['month'].min(), '->', df['month'].max())
df.head()


In [ ]:
cols=['median_price','tx_count','price_lag1','price_lag3_mean','target_logret','target_diff']
cols=[c for c in cols if c in df.columns]
fig,ax=plt.subplots(2,3,figsize=(12,7))
ax=ax.ravel()
for i,c in enumerate(cols):
    ax[i].hist(pd.to_numeric(df[c], errors='coerce').dropna(), bins=60)
    ax[i].set_title(c)
for j in range(i+1,6):
    ax[j].axis('off')
fig.tight_layout()
plt.show()


In [ ]:
# quick leakage check: random vs chrono with linear model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

feat=[c for c in ['price_lag1','price_lag3_mean','tx_lag1','month_sin','month_cos'] if c in df.columns]
yc='target_logret'

x=df[feat].apply(pd.to_numeric, errors='coerce').to_numpy(float)
y=pd.to_numeric(df[yc], errors='coerce').to_numpy(float)
keep=np.isfinite(x).all(axis=1) & np.isfinite(y)
x=x[keep]; y=y[keep]
d=df.loc[keep, ['month']].copy()

# random
xr,xt,yr,yt=train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)
m=LinearRegression().fit(xr,yr)
pr=m.predict(xt)

# chrono
months=np.array(sorted(pd.to_datetime(d['month']).dt.to_period('M').astype(str).unique()))
mtr=set(months[:int(0.8*len(months))])
mask=pd.to_datetime(d['month']).dt.to_period('M').astype(str).isin(mtr).to_numpy()
m2=LinearRegression().fit(x[mask], y[mask])
p2=m2.predict(x[~mask])

pd.DataFrame([
    {'split':'random','mse':mean_squared_error(yt,pr),'r2':r2_score(yt,pr)},
    {'split':'chrono','mse':mean_squared_error(y[~mask],p2),'r2':r2_score(y[~mask],p2)},
])
